# K-Nearest Neighbors: eine einfache Pipeline

KNN speichert Trainingsbeispiele und entscheidet für einen neuen Punkt anhand seiner nächsten Nachbarn. Weil alles von Abständen abhängt, gehören Skalierung und KNN gemeinsam in eine Pipeline.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.metrics import ConfusionMatrixDisplay,classification_report
from sklearn.model_selection import GridSearchCV,train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

X,y=make_classification(n_samples=600,n_features=2,n_redundant=0,n_informative=2,n_clusters_per_class=1,class_sep=1.15,random_state=42)
X[:,1]*=100
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.25,random_state=42,stratify=y)
plt.scatter(X[:,0],X[:,1],c=y,cmap="RdBu_r",edgecolor="white",s=28)
plt.xlabel("Merkmal 1"); plt.ylabel("Merkmal 2 - hundertfache Skala"); plt.show()

## Pipeline Schritt für Schritt

`StandardScaler.fit()` lernt Mittelwert und Standardabweichung ausschließlich aus den Trainingsdaten. `transform()` standardisiert damit neue Daten. KNNs `fit()` speichert diese Trainingspunkte und Labels; bei `predict()` werden erst die Nachbarn gesucht.

Die Pipeline verhindert Data Leakage und garantiert dieselbe Transformation bei Training, Test und späteren Vorhersagen.

In [ ]:
pipeline=Pipeline([
 ("skalierung",StandardScaler()),
 ("knn",KNeighborsClassifier(n_neighbors=5))])
pipeline.fit(X_train,y_train)
print("Test-Accuracy:",round(pipeline.score(X_test,y_test),3))
print(classification_report(y_test,pipeline.predict(X_test),target_names=["Klasse 0","Klasse 1"]))
ConfusionMatrixDisplay.from_estimator(pipeline,X_test,y_test,cmap="Blues"); plt.show()

skalierer=pipeline.named_steps["skalierung"]
knn=pipeline.named_steps["knn"]
punkt=X_test[[0]]; punkt_skaliert=skalierer.transform(punkt)
distanzen,indizes=knn.kneighbors(punkt_skaliert)
display(pd.DataFrame({"Distanz":distanzen[0],"Trainingsindex":indizes[0],"Klasse":y_train[indizes[0]]}))

## Warum Skalierung unverzichtbar ist

Ohne Skalierung dominiert hier Merkmal 2 allein wegen seines Zahlenbereichs. Die fachliche Information ist dadurch nicht größer.

In [ ]:
ohne=KNeighborsClassifier(n_neighbors=5).fit(X_train,y_train)
print("Ohne Skalierung:",round(ohne.score(X_test,y_test),3))
print("Mit Skalierung: ",round(pipeline.score(X_test,y_test),3))

def grenze(modell,ax,titel):
 x1=np.linspace(X[:,0].min(),X[:,0].max(),250); x2=np.linspace(X[:,1].min(),X[:,1].max(),250)
 xx,yy=np.meshgrid(x1,x2); z=modell.predict(np.c_[xx.ravel(),yy.ravel()]).reshape(xx.shape)
 ax.contourf(xx,yy,z,alpha=.35,cmap="RdBu_r"); ax.scatter(X_test[:,0],X_test[:,1],c=y_test,cmap="RdBu_r",edgecolor="white",s=24); ax.set_title(titel)
fig,ax=plt.subplots(1,2,figsize=(13,4)); grenze(ohne,ax[0],"Ohne Skalierung"); grenze(pipeline,ax[1],"Pipeline mit StandardScaler"); plt.show()

## Wichtige Parameter

| Parameter | Wirkung |
|---|---|
| `n_neighbors` | kleines k flexibel/rauschsensibel, großes k glatter |
| `weights` | `uniform` oder stärkere Stimmen naher Punkte mit `distance` |
| `metric` | Distanzbegriff, meist `minkowski` |
| `p` | bei Minkowski: 1 Manhattan, 2 euklidisch |
| `algorithm` | Suchverfahren: `auto`, `ball_tree`, `kd_tree`, `brute` |
| `leaf_size` | Laufzeit/Speicher für Baum-Suchverfahren, nicht Modellkomplexität |
| `n_jobs` | parallele Nachbarsuche, `-1` nutzt alle Kerne |

KNN lernt keine Koeffizienten. Tuning entscheidet, wie Nachbarschaft definiert und ausgewertet wird.

In [ ]:
ks=range(1,42,2); werte=[]
for k in ks:
 m=Pipeline([("scale",StandardScaler()),("knn",KNeighborsClassifier(n_neighbors=k))]).fit(X_train,y_train)
 werte.append({"k":k,"Training":m.score(X_train,y_train),"Test":m.score(X_test,y_test)})
pd.DataFrame(werte).plot(x="k",marker="o",figsize=(8,4)); plt.ylabel("Accuracy"); plt.grid(alpha=.3); plt.show()

grid={"knn__n_neighbors":[3,5,9,15,25],"knn__weights":["uniform","distance"],"knn__p":[1,2]}
suche=GridSearchCV(pipeline,grid,scoring="f1",cv=5,n_jobs=-1).fit(X_train,y_train)
print("Beste Parameter:",suche.best_params_); print("CV-F1:",round(suche.best_score_,3)); print("Test-Accuracy:",round(suche.score(X_test,y_test),3))

**Merksätze:** Skalierung gehört in die Pipeline. Kleines k kann overfitten, großes k underfitten. Die beste Konfiguration wird per Cross-Validation gewählt - der Testdatensatz bleibt bis zum Schluss unangetastet.